In [ ]:
import sys
sys.path.append('..')

import torch
import numpy as np
import random
from configs.config import Config
from data.dataset import BraTSDataset
from models.prototypical_segmentation import PrototypicalSegmentation
from training.prototypical_trainer import PrototypicalTrainer
from sklearn.model_selection import train_test_split
import os

# Set seeds for reproducibility
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

print(f"✓ Device: {Config.DEVICE}")

ModuleNotFoundError: No module named 'torch'

In [ ]:
# Load data
def get_patient_ids(data_path):
    dirs = [f.path for f in os.scandir(data_path) if f.is_dir()]
    return [d[d.rfind('/')+1:] for d in dirs]

all_ids = get_patient_ids(Config.TRAIN_DATASET_PATH)

# For testing, use subset (remove this line for full training)
all_ids = all_ids[:50]  # Start with 50 patients

# Split
train_val_ids, test_ids = train_test_split(all_ids, test_size=0.1, random_state=42)
train_ids, val_ids = train_test_split(train_val_ids, test_size=0.22, random_state=42)

print(f"✓ Train: {len(train_ids)}, Val: {len(val_ids)}, Test: {len(test_ids)}")

# Create datasets (no DataLoader needed for episodic training)
train_dataset = BraTSDataset(train_ids, Config.TRAIN_DATASET_PATH)
val_dataset = BraTSDataset(val_ids, Config.TRAIN_DATASET_PATH)
test_dataset = BraTSDataset(test_ids, Config.TRAIN_DATASET_PATH)

print(f"✓ Train samples: {len(train_dataset)}")
print(f"✓ Val samples: {len(val_dataset)}")

In [ ]:
# Create model
model = PrototypicalSegmentation(
    encoder_name='resnet34',
    in_channels=2,
    num_classes=4
).to(Config.DEVICE)

print(f"✓ Model created")
print(f"✓ Parameters: {sum(p.numel() for p in model.parameters()):,}")

# Load pretrained baseline encoder
checkpoint = torch.load('./checkpoints/best_model.pth')
model.unet.encoder.load_state_dict(checkpoint['model_state_dict']['unet.encoder'])
print("✓ Loaded pretrained encoder from baseline")

In [ ]:
# Cell 4: Test episode sampling
print("Testing episode sampling...")

trainer = PrototypicalTrainer(
    model=model,
    config=Config,
    train_dataset=train_dataset,
    val_dataset=val_dataset,
    test_loader=None  # Not needed for now
)

# Sample one episode
episode = trainer.sample_episode(train_dataset, k_shot=5, n_query=10)

print(f"✓ Support images: {episode['support_images'].shape}")  # (5, 2, 128, 128)
print(f"✓ Support masks: {episode['support_masks'].shape}")    # (5, 128, 128)
print(f"✓ Query images: {episode['query_images'].shape}")      # (10, 2, 128, 128)
print(f"✓ Query masks: {episode['query_masks'].shape}")        # (10, 128, 128)

In [ ]:
# Cell 5: Test prototype extraction
print("Testing prototype extraction...")

support_imgs = episode['support_images'].to(Config.DEVICE)
support_masks = episode['support_masks'].to(Config.DEVICE)

prototypes = model.compute_prototypes(support_imgs, support_masks, num_classes=4)

print(f"✓ Extracted {len(prototypes)} prototypes")
for i, proto in enumerate(prototypes):
    print(f"  Class {i} prototype: {proto.shape}")

In [ ]:
# Cell 6: Test prediction with prototypes
print("Testing prediction...")

query_imgs = episode['query_images'].to(Config.DEVICE)
query_masks = episode['query_masks'].to(Config.DEVICE)

predictions = model.predict_with_prototypes(query_imgs, prototypes)

print(f"✓ Predictions: {predictions.shape}")  # Should be (10, 4, 128, 128)

# Compute loss
loss_fn = smp.losses.DiceLoss(mode='multiclass')
loss = loss_fn(predictions, query_masks)
print(f"✓ Loss: {loss.item():.4f}")

In [ ]:
# Cell 7: TRAIN Prototypical Networks
print("="*60)
print("STARTING PROTOTYPICAL NETWORKS TRAINING")
print("="*60)

# Training configuration
NUM_EPISODES = 1000  # Start with 1000 episodes
K_SHOT = 5
N_QUERY = 10

trainer = PrototypicalTrainer(
    model=model,
    config=Config,
    train_dataset=train_dataset,
    val_dataset=val_dataset,
    test_loader=None
)

history = trainer.train(
    num_episodes=NUM_EPISODES,
    k_shot=K_SHOT,
    n_query=N_QUERY
)

print("✓ Training complete!")

In [ ]:
# Cell 8: Evaluate at different k values
print("Evaluating few-shot performance...")

k_shot_results = trainer.evaluate_k_shot(
    k_values=[1, 5, 10, 20],
    num_episodes=20  # 20 episodes per k value
)

# Print results
print("\nFew-Shot Performance:")
print("="*40)
for k, result in k_shot_results.items():
    print(f"k={k}: DICE = {result['mean']:.4f} ± {result['std']:.4f}")

# Save results
import json
with open(os.path.join(Config.RESULTS_DIR, 'prototypical_results.json'), 'w') as f:
    json.dump({k: {'mean': float(v['mean']), 'std': float(v['std'])} 
               for k, v in k_shot_results.items()}, f, indent=2)

In [ ]:
# Cell 9: Visualize learning curve
import matplotlib.pyplot as plt

k_values = [1, 5, 10, 20]
mean_dice = [k_shot_results[k]['mean'] for k in k_values]
std_dice = [k_shot_results[k]['std'] for k in k_values]

plt.figure(figsize=(10, 6))
plt.errorbar(k_values, mean_dice, yerr=std_dice, marker='o', capsize=5, 
             linewidth=2, markersize=8)
plt.xlabel('Number of Support Examples (k)', fontsize=12)
plt.ylabel('DICE Score', fontsize=12)
plt.title('Few-Shot Learning Curve: Prototypical Networks', fontsize=14)
plt.grid(True, alpha=0.3)
plt.xticks(k_values)
plt.ylim([0, 1])

# Add values on plot
for k, dice in zip(k_values, mean_dice):
    plt.text(k, dice + 0.02, f'{dice:.3f}', ha='center', fontsize=10)

plt.tight_layout()
plt.savefig(os.path.join(Config.RESULTS_DIR, 'prototypical_learning_curve.png'), dpi=150)
plt.show()

print("✓ Learning curve saved!")

In [ ]:
# Cell 10: Compare with baseline fine-tuning
def baseline_finetune_k_shot(base_model, train_dataset, test_loader, k=5):
    """Baseline: fine-tune on k examples"""
    import copy
    
    model = copy.deepcopy(base_model)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)
    loss_fn = smp.losses.DiceLoss(mode='multiclass')
    
    # Sample k examples
    indices = random.sample(range(len(train_dataset)), k)
    
    # Fine-tune
    model.train()
    for epoch in range(20):
        for idx in indices:
            sample = train_dataset[idx]
            img = sample['image'].unsqueeze(0).to(Config.DEVICE)
            mask = sample['mask'].unsqueeze(0).to(Config.DEVICE)
            
            optimizer.zero_grad()
            seg_out, _ = model(img)
            loss = loss_fn(seg_out, mask)
            loss.backward()
            optimizer.step()
    
    # Evaluate (simplified)
    model.eval()
    val_episode = trainer.sample_episode(val_dataset, k_shot=0, n_query=20)
    query_imgs = val_episode['query_images'].to(Config.DEVICE)
    query_masks = val_episode['query_masks'].to(Config.DEVICE)
    
    with torch.no_grad():
        pred, _ = model(query_imgs)
        dice_loss = loss_fn(pred, query_masks)
        dice = 1 - dice_loss.item()
    
    return dice

# Load your baseline model
from models.bu_net_multitask import BUNetMultiTask
baseline_model = BUNetMultiTask(
    encoder_name='resnet34',
    in_channels=2,
    num_classes=4,
    num_tumor_types=5
).to(Config.DEVICE)

checkpoint = torch.load('./checkpoints/best_model.pth')
baseline_model.load_state_dict(checkpoint['model_state_dict'])

# Compare baseline vs prototypical
print("\nComparing Prototypical vs Baseline Fine-tuning:")
print("="*50)

baseline_results = {}
for k in [1, 5, 10]:
    dice = baseline_finetune_k_shot(baseline_model, train_dataset, None, k=k)
    baseline_results[k] = dice
    print(f"Baseline k={k}: DICE = {dice:.4f}")
    print(f"Prototypical k={k}: DICE = {k_shot_results[k]['mean']:.4f}")
    improvement = k_shot_results[k]['mean'] - dice
    print(f"  → Improvement: {improvement:+.4f}\n")